In [1]:
from langgraph.graph import StateGraph , START , END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

In [2]:
load_dotenv()

model = ChatGroq(
    model = "openai/gpt-oss-20b"
)

In [21]:
class BlogState(TypedDict):
    title : str
    outline : str
    blog  : str
    evaluate : str

In [27]:
def create_outline(state:BlogState) -> BlogState:

    title = state['title']

    prompt = f'Give a detailed outline for blog on topic  {title}'
    outline = model.invoke(prompt)


    state['outline'] = outline
    return state


In [9]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline  = state['outline']

    prompt = f'Write a blog on the topic of title{title} using the following outline \n{outline}'
    blog = model.invoke(prompt )

    state ['blog'] = blog 
    return state


In [23]:
def evaluate(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    blog = state['blog']

    prompt = f'Act like an expert and rate this blog between 1-10:\n{blog}'

    rating = model.invoke(prompt).content

    state['evaluate'] = rating

    return state


In [25]:

graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

# compile
work_flow = graph.compile()

In [26]:
initial_state = {
    "title": "content creation",
    "outline": "",
    "blog": "",
    "evaluate":""
}

final_state = work_flow.invoke(initial_state)

print(final_state)

{'title': 'content creation', 'outline': AIMessage(content='## Blog Outline – “The Ultimate Guide to Content Creation”\n\n> **Audience:** Content creators, marketers, small‑business owners, and anyone who wants to produce high‑quality, audience‑engaging content consistently.  \n> **Goal:** Equip readers with a step‑by‑step framework, practical tools, and insider tips to turn ideas into publishable, measurable content.  \n\n---\n\n### 1. Title & Hook  \n- **Title:** *The Ultimate Guide to Content Creation: From Ideation to Impact*  \n- **Hook (first 2‑3 sentences):**  \n  - “Every brand that dominates its niche starts with a single piece of content that sparks conversation.”  \n  - “What if you could turn that spark into a repeatable, scalable process that delivers measurable ROI?”\n\n---\n\n### 2. Introduction  \n- **Why content matters today**  \n  - Stats: 70%+ of consumers prefer learning about a brand through articles vs ads.  \n  - The shift from “advertising” to “storytelling.”  

In [20]:
print(final_state['outline'])

content='# Blog Outline: **“The Ultimate Guide to Content Creation: From Ideation to Impact”**\n\n---\n\n## 1. Introduction  \n- **Hook**: A compelling statistic or anecdote that underscores the power of great content.  \n- **Purpose**: Explain what readers will gain—practical steps, tools, and mindset shifts.  \n- **Scope**: Briefly list the main stages of content creation (ideation, planning, production, distribution, measurement).  \n- **Target Audience**: Marketers, entrepreneurs, bloggers, social‑media managers, and anyone looking to improve their content game.\n\n---\n\n## 2. Why Content Creation Matters  \n- **Brand Visibility**: How content drives discovery and trust.  \n- **SEO & Organic Reach**: The role of high‑quality content in search rankings.  \n- **Lead Generation & Conversion**: Turning readers into customers.  \n- **Thought Leadership**: Establishing authority in niche markets.  \n- **Cost Efficiency**: Comparing content marketing ROI to traditional advertising.\n\n--